<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

##### 模块 2.6: 更多关于 ChiselTest
**上一节: [综合应用：FIR滤波器](2.5_exercise.ipynb)**<br>
**下一节: [生成器：参数](3.1_parameters.ipynb)**

## 动机
Chisel 团队一直在开发一个改进的测试框架。"ChiselTest" 提供了以下改进：

- 适用于单元测试和系统集成测试
- 设计用于可组合的抽象和分层
- 高度可用，通过使其尽可能简单、无痛（避免样板代码和其他无意义的内容）和有用，鼓励编写单元测试

### 计划中
- 能够针对多个后端和模拟器（如果测试向量不是静态的，可能需要链接到 Scala，或者在合成到 FPGA 时使用有限的测试构造 API 子集）
- 将包含在基础 chisel3 中，以避免打包和依赖问题


## 设置

In [1]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

path: String = "/Users/zhaochanglong/Documents/chisel-bootcamp-zh/source/load-ivy.sc"

In [2]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.experimental.BundleLiterals._
import chisel3.tester._
import chisel3.tester.RawTester.test

import chisel3._

import chisel3.util._

import chisel3.experimental._

import chisel3.experimental.BundleLiterals._

import chisel3.tester._

import chisel3.tester.RawTester.test

>这个训练营需要与你在其他地方看到的 Chisel 导入略有不同。`import chisel3.tester.RawTester.test` 引入了下面版本的 `test(...)`，这是专门为训练营设计的。

---
# Basic 测试器 实现

ChiselTest starts with the same basic operations as iotesters. Here's a brief 总结 of the basic
functionality mapping between the older iotesters and the new ChiselTest

|        | iotesters             | ChiselTest            |
| :----  | :---                  | :---                |
| 注入   | 注入(c.io.in1, 6)     | c.io.in1.注入(6.U)    |
| peek   | peek(c.io.out1)       | c.io.out1.peek()      |
| 期望 | 期望(c.io.out1, 6)  | c.io.out1.期望(6.U) |
| 步进   | 步进(1)               | c.io.时钟.步进(1)  |
| initiate | Driver.execute(...) { c => | 测试(...) { c => |


让我们从查看 2.1 中的简单直通模块开始

In [3]:
// Chisel Code, but pass in a parameter to set widths of ports
class PassthroughGenerator(width: Int) extends Module { 
  val io = IO(new Bundle {
    val in = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  io.out := io.in
}

defined class PassthroughGenerator

使用旧风格的测试就像下面这样：

```scala
val testResult = Driver(() => new Passthrough()) {
  c => new PeekPokeTester(c) {
    poke(c.io.in, 0)     // 设置输入的值为 0
    expect(c.io.out, 0)  // 断言输出为 0
    poke(c.io.in, 1)     // 设置输入的值为 1
    expect(c.io.out, 1)  // 断言输出为 1
    poke(c.io.in, 2)     // 设置输入的值为 2
    expect(c.io.out, 2)  // 断言输出为 2
  }
}
assert(testResult)   // Scala Code: if testResult == false, will throw an error
println("SUCCESS!!") // Scala Code: if we get here, our tests passed!
```



In [4]:
test(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}

Elaborating design...
Done elaborating.
test PassthroughGenerator Success: 0 tests passed in 2 cycles in 0.016424 seconds 121.77 Hz


>为了说明 ChiselTest 是如何对时钟进行前进操作的，我们可以在之前的示例中添加一些`步进`操作。

In [5]:
test(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.clock.step(1)    // advance the clock
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.clock.step(1)    // advance the clock
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.clock.step(1)    // advance the clock
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}

Elaborating design...
Done elaborating.
test PassthroughGenerator Success: 0 tests passed in 5 cycles in 0.004027 seconds 1241.71 Hz


---
## What to notice in the above 示例

ChiselTest's `测试` 方法 requires a bit less boiler plate. What was the `PeekPokeTester` is now
built into the process.

The `注入` and `期望` methods are now part of each individual `io` element.
This gives important hints to the 测试器 to make better checking of types.
The `peek` and `步进` operations are also now methods on `io` elements.

Another difference is that values poked and expected are Chisel literals.
Although pretty simple here, it also provides stronger checking in more advanced and interesting 示例.
This will be further enhanced with coming improvements in the ability to specify `束` literals



# Modules with Decoupled Interfaces
In this section we will look at some of the tester2's tools for working with `Decoupled` interfaces.
`Decoupled` takes a Chisel data 类型 and provides it with `ready` and `valid` signals.
ChiselTest provides some nice tools for automating and reliably testing these interfaces.

## A queue 示例
The `QueueModule` passes through data whose 类型 is determined by `ioType`. There are `entries` state elements inside the `QueueModule` meaning it can hold that many elements before it exerts backpressure.

In [ ]:
class QueueModule[T <: Data](ioType: T, entries: Int) extends MultiIOModule {
  val in = IO(Flipped(Decoupled(ioType)))
  val out = IO(Decoupled(ioType))
  out <> Queue(in, entries)
}

## EnqueueNow and expectDequeueNow
*ChiselTest* has some built in methods for dealing with circuits with Decoupled interfaces in the IOs. 在这个例子中 we will see how to insert and extract values from the `queue`. 

| 方法 | description |
| :---   | :---        |
| enqueueNow | Add (enqueue) one element to a `Decoupled` 输入 interface |
| expectDequeueNow | Removes (dequeues) one element from a `Decoupled` 输出 interface |
---


>Note: There is some required boiler plate `initSource`, `setSourceClock`, etc 也就是说 necessary to ensure that the `ready` and `valid` fields are
all initialized correctly at the beginning of the 测试.


In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // Example testsequence showing the use and behavior of Queue
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(200){ i => i.U }

    testVector.zip(testVector).foreach { case (in, out) =>
      c.in.enqueueNow(in)
      c.out.expectDequeueNow(out)
    }
}

## EnqueueSeq and DequeueSeq 
Now we are going to introduce two new methods that deal with enqueuing and dequeuing operations in single operations.

| 方法 | description |
| :---   | :---        |
| enqueueSeq | Continues to add (enqueue) elements from the `Seq` to a `Decoupled` 输入 interface, one at a time, until the sequence is exhausted |
| expectDequeueSeq | Removes (dequeues) elements from a `Decoupled` 输出 interface, one at a time, and compares each one to the next element of the `Seq` |
---
> Note: The 示例 below works fine but, as written, the `enqueueSeq` must finish before the `expectDequeueSeq` can begin. This 示例 would fail if the `testVector`'s size is made larger than the queue depth, because the queue would fill up and not be able to complete the `enqueueSeq`. Try it yourself to see what the failure looks like. In the next section we will show to construct this 类型 of 测试 properly.


In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // Example testsequence showing the use and behavior of Queue
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(100){ i => i.U }

    c.in.enqueueSeq(testVector)
    c.out.expectDequeueSeq(testVector)
}

> One more important take away from the last section is that the functions we just saw, `enqueueNow`, 
`enqueueSeq`, `expectDequeueNow`, and `expectDequeueSeq` are not complicated special case logic in ChiselTest.
Rather they are 示例 of the kinds of harness building that ChiselTest encourages you to build from the ChiselTest primitives. To see how these methods are implemented check out [TestAdapters.scala](https://github.com/ucb-bar/Chisel-testers2/blob/d199c5908828d0be5245f55fce8a872b2afb314e/src/main/scala/chisel3/测试器/TestAdapters.scala)

# Fork and Join in ChiselTest

In this section we will look at running sections of a unit 测试 concurrently. In order to do this we will introduce two new features of testers2.

| 方法 | description |
| :---   | :---        |
| fork   | launches a concurrent code block, additional forks can be run concurrently to this one via the .fork appended to end of the code block of the preceeding fork |
| join | re-unites multiple related forks back into the calling thread |
---

In the 示例 below two `fork`s are chained together, and then `join`ed. In the first `fork` block the `enqueueSeq` will continue to add elements until exhausted. The second `fork` block will `expectDequeueSeq` on each cycle when data is available.

>The threads created by fork are run in a deterministic order, largely according to their order as specified in code, and certain bug-prone operations that depend on other threads are forbidden with runtime checks. 


In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // Example testsequence showing the use and behavior of Queue
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(300){ i => i.U }

    fork {
        c.in.enqueueSeq(testVector)
    }.fork {
        c.out.expectDequeueSeq(testVector)
    }.join()
}

## Using Fork and Join with GCD
In this section we will use the fork join methods to implement tests of *Greatest Common Denominator* **GCD**.
Let's start by defining our IO bundles. We are going to add a bit of boiler plate here to allow us to use `束` *literals*. As the comments say, it is hoped that we will soon have support for autogeneration of the 字面量 support code.

In [ ]:
class GcdInputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
}

In [ ]:
class GcdOutputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
  val gcd    = UInt(w.W)
}

Now let's look at a *Decoupled* version of **GCD**. We've modified it a bit here to use the `Decoupled` wrapper that adds a `ready` and a `valid` signal to the 输入 and 输出 束. The `Flipped` wrapper takes the `Decoupled` `GcdInputBundle` which by default is created as an 输出 and converts each field to the opposite direction (recursively). The data elements of the bundled arguments to `Decoupled` are placed in the top level field `bits`. 

In [ ]:
/**
  * Compute GCD using subtraction method.
  * Subtracts the smaller of registers x and y from the larger until register y is zero.
  * value input register x is then the Gcd
  * returns a packet of information with the two input values and their GCD
  */
class DecoupledGcd(width: Int) extends MultiIOModule {

  val input = IO(Flipped(Decoupled(new GcdInputBundle(width))))
  val output = IO(Decoupled(new GcdOutputBundle(width)))

  val xInitial    = Reg(UInt())
  val yInitial    = Reg(UInt())
  val x           = Reg(UInt())
  val y           = Reg(UInt())
  val busy        = RegInit(false.B)
  val resultValid = RegInit(false.B)

  input.ready := ! busy
  output.valid := resultValid
  output.bits := DontCare

  when(busy)  {
    // during computation keep subtracting the smaller from the larger
    when(x > y) {
      x := x - y
    }.otherwise {
      y := y - x
    }
    when(y === 0.U) {
      // when y becomes zero computation is over,
      // signal valid data to output if the output is ready
      output.bits.value1 := xInitial
      output.bits.value2 := yInitial
      output.bits.gcd := x
      output.valid := true.B
      busy := ! output.ready
    }
  }.otherwise {
    when(input.valid) {
      // valid data available and no computation in progress, grab new values and start
      val bundle = input.deq()
      x := bundle.value1
      y := bundle.value2
      xInitial := bundle.value1
      yInitial := bundle.value2
      busy := true.B
      resultValid := false.B
    }
  }
}

我们的测试看起来与之前的队列测试基本相同。
但这里还有更多事情发生，因为计算需要多个周期，所以在计算每个 GCD 时，输入入队过程会被阻塞。
好消息是，这方面的测试端很简单，并且在不同的解耦电路中保持一致。

这里还介绍了新的 Chisel3 `Bundle` 字面量表示法。考虑这一行：
```scala
new GcdInputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U)
```
上面定义的 `GcdInputBundle` 有两个字段 `value1` 和 `value2`。
我们通过首先创建一个 Bundle，然后调用其 `.Lit` 方法来创建一个 Bundle 字面量。
该方法接受一个键/值对的变量参数列表，其中键（例如 `_.value1`）是字段名，值（例如 x.U）是一个 Chisel 硬件字面量，Scala `Int` x 被转换为 Chisel `UInt` 字面量。
字段名前面的 `_.` 是必要的，用于将名称值绑定到 Bundle 内部。

>这可能不是完美的表示法，但在广泛的开发讨论中，它被视为在最小化样板代码和 Scala 中可用的表示法限制之间达到最佳平衡。


In [ ]:
test(new DecoupledGcd(16)) { dut =>
  dut.input.initSource().setSourceClock(dut.clock)
  dut.output.initSink().setSinkClock(dut.clock)

  val testValues = for { x <- 1 to 10; y <- 1 to 10} yield (x, y)
  val inputSeq = testValues.map { case (x, y) =>
    (new GcdInputBundle(16)).Lit(_.value1 -> x.U, _.value2 -> y.U)
  }
  val resultSeq = testValues.map { case (x, y) =>
    new GcdOutputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U, _.gcd -> BigInt(x).gcd(BigInt(y)).U)
  }

  fork {
    dut.input.enqueueSeq(inputSeq)
  }.fork {
    for (expected <- resultSeq) {
      dut.output.expectDequeue(expected)
      dut.clock.step(5) // wait some cycles before receiving the next output to create backpressure
    }
  }.join()
}


---
# 你完成了！

[返回顶部。](#top)